# 05 - Instrumental variables

Every method so far has needed the confounders to be measured. An instrument
escapes that requirement: a variable that shifts treatment, is unrelated to the
outcome except through treatment, and is as good as randomly assigned.

The escape is real, and it is expensive. The instrument identifies the effect
only among units whose treatment responds to it, and that subpopulation is
neither observable nor necessarily representative.

## Causal question

Treatment take-up is driven partly by an encouragement that we can treat as
random, and partly by characteristics we cannot observe. What is the effect of
treatment on the outcome?

## Data and design

- **Unit of analysis:** one individual.
- **Instrument:** `instrument`, binary encouragement, randomly assigned.
- **Treatment:** `treatment`, binary, influenced by the instrument and by
  `hidden_confounder`.
- **Outcome:** `outcome`.
- **Covariate:** `x`, observed.
- **`hidden_confounder`:** present in the data so we can demonstrate the
  problem, and never used by any estimator — that is the point of it.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab.data_generators import make_iv_data
from causal_inference_lab.estimators import difference_in_means, g_computation_ate
from causal_inference_lab.instrumental_variables import instrumental_variables_ate

dataset = make_iv_data(n=5_000, seed=99)
data = dataset.data

print(f"observations:  {len(data):,}")
print(f"true effect:   {dataset.true_ate:.3f}")
print(f"encouraged:    {data['instrument'].mean():.1%}")
print(f"treated:       {data['treatment'].mean():.1%}")
print()
take_up = data.groupby("instrument")["treatment"].mean()
print("treatment take-up by encouragement:")
print(f"  not encouraged: {take_up.loc[0]:.3f}")
print(f"  encouraged:     {take_up.loc[1]:.3f}")
print(f"  difference:     {take_up.loc[1] - take_up.loc[0]:.3f}  <- the compliers")

**Interpretation.** The encouragement raises take-up by 0.26. That difference
defines the complier group: roughly a quarter of the sample takes the treatment
because they were encouraged. Everyone else either takes it regardless or
refuses regardless, and contributes nothing to the identification.

## Estimand

The **local average treatment effect (LATE)**: the average effect among
compliers.

This is not the ATE, and the distinction is not pedantry. Compliers here are
about 26% of the sample. If their effects differ systematically from
never-takers and always-takers — which is exactly what one expects when people
select into treatment — the LATE does not generalise. The result object reports
`estimand="LATE"` for this reason.

## Identification assumptions

1. **Relevance.** The instrument genuinely shifts treatment. Checkable, via the
   first-stage F-statistic.
2. **Exclusion restriction.** The instrument affects the outcome *only* through
   treatment. Untestable, and usually the weakest link.
3. **Independence.** The instrument is as good as randomly assigned with respect
   to potential outcomes.
4. **Monotonicity.** No defiers — nobody takes the treatment *because* they were
   not encouraged.

Only the first is checkable from data. An encouragement that also affects the
outcome directly — by raising awareness, say — violates the exclusion
restriction, and no diagnostic in this notebook would detect it.

## Estimation

Two-stage least squares, alongside the estimators that ignore the hidden
confounder.

In [ ]:
naive = difference_in_means(data)
adjusted = g_computation_ate(data, covariates=["x"])
iv = instrumental_variables_ate(data)

print(f"true effect:                     {dataset.true_ate:.3f}")
print()
print(f"naive difference in means:       {naive.estimate:.3f}   ({naive.estimand})")
print(f"adjusted for observed x:         {adjusted.estimate:.3f}   ({adjusted.estimand})")
print(f"two-stage least squares:         {iv.effect.estimate:.3f}   ({iv.effect.estimand})")

**Interpretation.** The naive contrast gives 3.60 and adjusting for the observed
covariate barely helps at 3.48, because the confounding lives in a variable
neither estimator can see. Both overstate the true 2.50 by around 40%.

The instrumental variables estimate is 2.37 — close to the truth, obtained
without ever measuring the confounder. That is the trade IV offers: it buys
freedom from unmeasured confounding by spending the exclusion restriction, which
is itself an untestable assumption. You are not removing assumptions, you are
exchanging one for another that may be easier to defend.

## Diagnostics

Instrument strength is the one assumption the data can speak to. The first-stage
F-statistic measures how much of the variation in treatment the instrument
explains; the conventional threshold is 10.

In [ ]:
print(f"first-stage F-statistic:  {iv.first_stage_f_stat:.1f}")
print(f"flagged as weak:          {iv.instrument_is_weak}")
print(f"first-stage R-squared:    {iv.first_stage_r2:.3f}")
print(f"instrument coefficient:   {iv.instrument_coefficient:.3f}")

**Interpretation.** An F of 380 is far above the threshold, so weakness is not a
concern here. Note the R-squared of 0.071 alongside it: the instrument explains
only 7% of the variation in treatment, and that is entirely normal. F and
R-squared answer different questions, and it is F that governs the bias.

The threshold deserves more scrutiny than it usually gets. We can weaken the
instrument deliberately — scrambling a growing share of its values — and watch
what the estimate does.

In [ ]:
rng = np.random.default_rng(3)
rows = []
for scrambled in (0.0, 0.7, 0.9, 0.97):
    weakened = data.copy()
    mask = rng.random(len(weakened)) < scrambled
    weakened.loc[mask, "instrument"] = rng.binomial(1, 0.5, int(mask.sum()))

    run = instrumental_variables_ate(weakened)
    shift = weakened.groupby("instrument")["treatment"].mean()
    rows.append(
        {
            "scrambled": f"{scrambled:.0%}",
            "F-stat": run.first_stage_f_stat,
            "flagged weak": run.instrument_is_weak,
            "estimate": run.effect.estimate,
            "take-up shift": shift.loc[1] - shift.loc[0],
        }
    )

print(f"true effect: {dataset.true_ate:.3f}\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

**Interpretation.** The estimates become erratic as the instrument weakens:
2.37, then 1.32, then 2.67, then 1.66, against a truth of 2.50. There is no
orderly degradation to reason about.

The row worth dwelling on is the second. At 70% scrambled the F-statistic is
31.9 — three times the conventional threshold, comfortably "strong" — and the
estimate is 1.32, off by 47%. Passing F > 10 is not a guarantee of a usable
estimate.

Two mechanisms are at work and honesty requires separating them. Weak
instruments bias IV towards the confounded OLS estimate and inflate variance.
But scrambling also *changes who the compliers are*, and the LATE is defined by
that group — so part of the movement is a genuinely different estimand rather
than error. Both are reasons to distrust an estimate built on a marginal
instrument.

## Uncertainty

Bootstrapping the whole 2SLS procedure gives an interval that reflects both
stages.

In [ ]:
boot_rng = np.random.default_rng(99)
estimates = []
for _ in range(300):
    resample = data.iloc[boot_rng.integers(0, len(data), len(data))]
    estimates.append(instrumental_variables_ate(resample).effect.estimate)

estimates = np.array(estimates)
lower, upper = np.percentile(estimates, [2.5, 97.5])

print(f"IV estimate:   {iv.effect.estimate:.3f}")
print(f"95% interval:  [{lower:.3f}, {upper:.3f}]")
print(f"standard error:{estimates.std():.3f}")
print(f"true effect:   {dataset.true_ate:.3f}")
print(f"covers truth:  {lower <= dataset.true_ate <= upper}")
print(f"\ninterval width: {upper - lower:.3f}")

**Interpretation.** The interval covers the true effect and is noticeably wide —
much wider than the AIPW intervals in notebooks 01 and 02, on data of the same
size.

That width is the real price of the design. IV extracts its answer from the
complier subpopulation only, so the effective sample is a fraction of the
nominal 5,000. Buying robustness to unmeasured confounding costs precision, and
the cost is visible here rather than hidden.

## Limitations

- **The estimand is the LATE, not the ATE.** It applies to the roughly 26% of
  units whose treatment responds to the encouragement. Nothing here supports a
  claim about never-takers or always-takers.
- **The exclusion restriction is untestable.** If the encouragement affects
  outcomes by any route other than treatment, the estimate is biased and no
  diagnostic in this notebook reacts.
- **Monotonicity is assumed.** Defiers would invalidate the LATE
  interpretation, and are not detectable.
- **F > 10 is a rule of thumb, not a guarantee.** The table above shows an
  estimate off by 47% at F = 31.9.
- **Wide intervals.** The design is inefficient by construction; distinguishing
  moderate effects from zero needs considerably more data than a
  selection-on-observables approach would.
- **Synthetic data with a single binary instrument.** Real applications often
  have several instruments of debatable validity, which raises questions of
  overidentification not covered here.